In [1]:
import pandas as pd

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Algorithm.ExtremeLearningMachine import ExtremeLearningMachine
from Pipeline.Methodology.EvaluationMatrix import EvaluationMatrix
from Pipeline.Global.GlobalSetting import GlobalSetting

In [2]:
model_configs = GlobalSetting.get_model_configs()

In [3]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train_scaled.shape[1]

In [4]:
x_test_scaled   = gallstone_dataset.x_test_scaled
y_test          = gallstone_dataset.y_test
x_train_scaled  = gallstone_dataset.x_train_scaled
y_train         = gallstone_dataset.y_train

In [5]:
testing_results = []
for config in model_configs:
    for seed in GlobalSetting.elm_initial_state_range:
        elm = ExtremeLearningMachine(
            features_size   = features_size,
            hidden_size     = config["Hidden_Nodes"],
            activation_function     = config["Activation"],
            regularization_lambda   = config["Lambda_Value"]
        )
        elm.initialize_random_weights(random_seed = seed)

        elm.fit(x_train_scaled.values, y_train.values)
        y_pred = elm.predict(x_test_scaled.values)

        evaluation = EvaluationMatrix(y_test, y_pred)
        print(f"--- Config: {config['Hidden_Nodes']} Nodes | Seed: {seed} ---")
        print(evaluation.get_report())

        metrics = evaluation.get_all_metrics()
        test_result = {
            "Model_Type"   : config.get('Model_Types', 'Unknown'),
            "Hidden_Nodes" : config['Hidden_Nodes'],
            "Lambda_Value" : config['Lambda_Value'],
            "Activation"   : config['Activation'].__name__,
            "Seed"         : seed
        }
        test_result.update(metrics)
        testing_results.append(test_result)
        print("\n")

--- Config: 48 Nodes | Seed: 161 ---
{'Counts': {'TP': np.int64(26), 'TN': np.int64(25), 'FP': np.int64(7), 'FN': np.int64(6)}, 'Metrics': {'Accuracy': np.float64(0.7969), 'Precision': 0.7879, 'Recall': 0.8125, 'NPV': 0.8065, 'Specificity': 0.7812, 'F1-Score': 0.8, 'F2-Score': 0.8075, 'Bal Accuracy': 0.7969, 'MCC': 0.594}}


--- Config: 48 Nodes | Seed: 162 ---
{'Counts': {'TP': np.int64(24), 'TN': np.int64(24), 'FP': np.int64(8), 'FN': np.int64(8)}, 'Metrics': {'Accuracy': np.float64(0.75), 'Precision': 0.75, 'Recall': 0.75, 'NPV': 0.75, 'Specificity': 0.75, 'F1-Score': 0.75, 'F2-Score': 0.75, 'Bal Accuracy': 0.75, 'MCC': 0.5}}


--- Config: 48 Nodes | Seed: 163 ---
{'Counts': {'TP': np.int64(22), 'TN': np.int64(26), 'FP': np.int64(6), 'FN': np.int64(10)}, 'Metrics': {'Accuracy': np.float64(0.75), 'Precision': 0.7857, 'Recall': 0.6875, 'NPV': 0.7222, 'Specificity': 0.8125, 'F1-Score': 0.7333, 'F2-Score': 0.7051, 'Bal Accuracy': 0.75, 'MCC': 0.504}}


--- Config: 48 Nodes | Seed: 164 -

In [6]:
# 1. 将字典列表转换为 DataFrame
df_results = pd.DataFrame(testing_results)

# 2. 定义哪些列代表一个唯一的 "Algorithm" 配置
group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation"]

# 3. 动态提取所有需要聚合的指标列（排除配置列和 Seed）
metric_cols = [col for col in df_results.columns if col not in group_cols + ["Seed"]]

# 4. 构造聚合字典：对每个指标求平均值(mean)和标准差(std)
agg_funcs = {metric: ['mean', 'std', 'max' , 'min'] for metric in metric_cols}

# 5. 执行 Group By
summary_df = df_results.groupby(group_cols).agg(agg_funcs).reset_index()

# 将多级表头 (例如 'Accuracy', 'mean') 拼接为单级表头 'Accuracy_mean'
summary_df.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df.columns.values
]

summary_df

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy_mean,Accuracy_std,Accuracy_max,Accuracy_min,Precision_mean,Precision_std,...,F2-Score_max,F2-Score_min,Bal Accuracy_mean,Bal Accuracy_std,Bal Accuracy_max,Bal Accuracy_min,MCC_mean,MCC_std,MCC_max,MCC_min
0,Best_Hidden_Nodes,48,0.000000,sigmoid,0.760938,0.039390,0.828125,0.68750,0.770900,0.043020,...,0.848485,0.645161,0.760938,0.039390,0.828125,0.68750,0.523595,0.078820,0.656571,0.375000
1,Best_Lambda,48,0.007812,sigmoid,0.774479,0.035223,0.843750,0.71875,0.786619,0.033724,...,0.864198,0.677419,0.774479,0.035223,0.843750,0.71875,0.550651,0.069883,0.688847,0.437500
2,Best_Size_and_Lambda,48,0.007812,sigmoid,0.774479,0.035223,0.843750,0.71875,0.786619,0.033724,...,0.864198,0.677419,0.774479,0.035223,0.843750,0.71875,0.550651,0.069883,0.688847,0.437500
3,Grid_Combination,93,0.062500,sigmoid,0.801562,0.027265,0.859375,0.75000,0.809654,0.038535,...,0.864198,0.750000,0.801562,0.027265,0.859375,0.75000,0.604027,0.055569,0.719101,0.500000
4,Grid_Optimization,37,0.077887,sigmoid,0.781771,0.037938,0.843750,0.71875,0.792528,0.046431,...,0.884146,0.673077,0.781771,0.037938,0.843750,0.71875,0.565889,0.076638,0.692935,0.438357


In [7]:
GlobalSetting.save_dataframe_to_record(summary_df,"Model_Testing_Result.csv")

[I/O Trace] Record exported successfully: ../../Storage/Record\Model_Testing_Result.csv
